In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data_read_to_use.csv")

print(df.shape)
print(df.columns.tolist())
df.head()

(835, 12)
['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight', 'Year', 'City', 'Sport', 'Event', 'Medal', 'Clean_Name']


,ID,Name,Sex,Age,Height,Weight,Year,City,Sport,Event,Medal,Clean_Name
0,512,Shiny Kurisingal Abraham-Wilson,F,19.0,167.0,53.0,1984,Los Angeles,Athletics,Women's 800 metres,No Medal,Shiny Kurisingal Abraham-Wilson
1,512,Shiny Kurisingal Abraham-Wilson,F,19.0,167.0,53.0,1984,Los Angeles,Athletics,Women's 4 x 400 metres Relay,No Medal,Shiny Kurisingal Abraham-Wilson
2,512,Shiny Kurisingal Abraham-Wilson,F,23.0,167.0,53.0,1988,Seoul,Athletics,Women's 800 metres,No Medal,Shiny Kurisingal Abraham-Wilson
3,512,Shiny Kurisingal Abraham-Wilson,F,23.0,167.0,53.0,1988,Seoul,Athletics,Women's 4 x 400 metres Relay,No Medal,Shiny Kurisingal Abraham-Wilson
4,512,Shiny Kurisingal Abraham-Wilson,F,27.0,167.0,53.0,1992,Barcelona,Athletics,Women's 800 metres,No Medal,Shiny Kurisingal Abraham-Wilson


In [3]:
## missing values inspection
# ideally non zero because of proper cleaning 😅😅
df.info()

print("\nMissing values:")
print(df.isna().sum())

print("\nExact duplicate rows:")
print(df.duplicated().sum())


<class 'pandas.DataFrame'>
RangeIndex: 835 entries, 0 to 834
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          835 non-null    int64  
 1   Name        835 non-null    str    
 2   Sex         835 non-null    str    
 3   Age         835 non-null    float64
 4   Height      835 non-null    float64
 5   Weight      835 non-null    float64
 6   Year        835 non-null    int64  
 7   City        835 non-null    str    
 8   Sport       835 non-null    str    
 9   Event       835 non-null    str    
 10  Medal       835 non-null    str    
 11  Clean_Name  835 non-null    str    
dtypes: float64(3), int64(2), str(7)
memory usage: 78.4 KB

Missing values:
ID            0
Name          0
Sex           0
Age           0
Height        0
Weight        0
Year          0
City          0
Sport         0
Event         0
Medal         0
Clean_Name    0
dtype: int64

Exact duplicate rows:
0


In [4]:
## I intentionally left same athlete while cleaning data if there is at least one metric (physical aspect)
## differ so now I need to verify that

In [5]:
print("Total records:", len(df))
print("Unique athletes:", df["ID"].nunique())

Total records: 835
Unique athletes: 522


In [7]:
repeated = df[df.duplicated(subset=["ID"], keep=False)]

print("Records belonging to repeated athletes:", len(repeated))

repeated.sort_values(["ID", "Year"]).head(20)

Records belonging to repeated athletes: 503


,ID,Name,Sex,Age,Height,Weight,Year,City,Sport,Event,Medal,Clean_Name
0,512,Shiny Kurisingal Abraham-Wilson,F,19.0,167.0,53.0,1984,Los Angeles,Athletics,Women's 800 metres,No Medal,Shiny Kurisingal Abraham-Wilson
1,512,Shiny Kurisingal Abraham-Wilson,F,19.0,167.0,53.0,1984,Los Angeles,Athletics,Women's 4 x 400 metres Relay,No Medal,Shiny Kurisingal Abraham-Wilson
2,512,Shiny Kurisingal Abraham-Wilson,F,23.0,167.0,53.0,1988,Seoul,Athletics,Women's 800 metres,No Medal,Shiny Kurisingal Abraham-Wilson
3,512,Shiny Kurisingal Abraham-Wilson,F,23.0,167.0,53.0,1988,Seoul,Athletics,Women's 4 x 400 metres Relay,No Medal,Shiny Kurisingal Abraham-Wilson
4,512,Shiny Kurisingal Abraham-Wilson,F,27.0,167.0,53.0,1992,Barcelona,Athletics,Women's 800 metres,No Medal,Shiny Kurisingal Abraham-Wilson
5,512,Shiny Kurisingal Abraham-Wilson,F,31.0,167.0,53.0,1996,Atlanta,Athletics,Women's 4 x 400 metres Relay,No Medal,Shiny Kurisingal Abraham-Wilson
6,663,Sharath Kamal Achanta,M,22.0,186.0,85.0,2004,Athina,Table Tennis,Men's Singles,No Medal,Sharath Kamal Achanta
7,663,Sharath Kamal Achanta,M,26.0,186.0,85.0,2008,Beijing,Table Tennis,Men's Singles,No Medal,Sharath Kamal Achanta
8,663,Sharath Kamal Achanta,M,34.0,186.0,85.0,2016,Rio de Janeiro,Table Tennis,Men's Singles,No Medal,Sharath Kamal Achanta
10,1366,Jitendarjit Singh Ahluwalia,M,25.0,179.0,69.0,1980,Moskva,Equestrianism,"Mixed Three-Day Event, Individual",No Medal,Jitendarjit Singh Ahluwalia


In [8]:
## the key point is athlete belong to differnet events or years are not our major concern the physical aspects is

In [10]:
measurements = ["Height", "Weight"]

variation = df.groupby("ID")[measurements].nunique()

for col in measurements:
    print(f"Athletes with different {col} values:",
          (variation[col] > 1).sum())

Athletes with different Height values: 0
Athletes with different Weight values: 0


In [12]:
# Number of distinct athlete-year combinations
print(df[["ID", "Year"]].drop_duplicates().shape[0])

# Athlete-year combinations with multiple records
counts = df.groupby(["ID", "Year"]).size()

print(counts[counts > 1])

732
ID      Year
512     1984    2
        1988    2
1366    1980    2
3539    1988    2
4000    1964    2
               ..
125624  2000    2
        2004    2
        2008    2
126823  2012    2
132177  2016    2
Length: 94, dtype: int64


In [13]:
## checking diff measurements in the same year 
variation = df.groupby(["ID", "Year"])[
    ["Age", "Height", "Weight"]
].nunique(dropna=False)

conflicts = variation[
    (variation > 1).any(axis=1)
]

print("Athlete-year groups with different measurements:", len(conflicts))
print(conflicts)

Athlete-year groups with different measurements: 0
Empty DataFrame
Columns: [Age, Height, Weight]
Index: []


In [14]:
## copying data frame
df_clean = df.copy()

In [15]:
df_clean['BMI'] = (df_clean['Weight'] / (df_clean['Height']/100)**2)

In [16]:
df_clean[["ID", "Year", "Height", "Weight", "BMI"]].head()

,ID,Year,Height,Weight,BMI
0,512,1984,167.0,53.0,19.003908
1,512,1984,167.0,53.0,19.003908
2,512,1988,167.0,53.0,19.003908
3,512,1988,167.0,53.0,19.003908
4,512,1992,167.0,53.0,19.003908


In [17]:
## stats so far
columns = ["Age", "Height", "Weight", "BMI"]

print(df_clean[columns].describe())

              Age      Height      Weight         BMI
count  835.000000  835.000000  835.000000  835.000000
mean    25.643114  171.704192   67.695210   22.805176
std      5.390689    9.603777   14.366965    3.455087
min     15.000000  137.000000   43.000000   17.319120
25%     22.000000  165.000000   58.000000   20.438166
50%     25.000000  172.000000   65.000000   22.203339
75%     28.000000  178.000000   74.000000   24.175156
max     66.000000  200.000000  125.000000   39.183673


In [19]:
# more roboust data focusing on question 1
df_clean.to_csv("q1_working.csv", index=False)